# Cybersecurity Mean-Field Control Benchmark

Reference: Meunier, Pham & Reisinger, discrete-space benchmarks, Section "Cybersecurity Example" (`files/reference/discrete_benchmarks.tex`), adapted from Kolokoltsov & Bensoussan's mean-field game of botnet defense via Carmona et al.'s mean-field control formulation.

**Model.** A large population of computers, each in one of four states $\mathcal X=\{\mathrm{DI},\mathrm{DS},\mathrm{UI},\mathrm{US}\}$ (defended/undefended $\times$ infected/susceptible). Action space $\mathcal A=\{0,1\}$: $0$ keeps the current protection level, $1$ switches it (rate $\lambda_\mathrm{sw}$). Infection spreads at endogenous rates depending on the population's own infected fractions:
$$\iota_D(\mu)=v_H q_\mathrm{inf}^D+\beta_{DD}\mu(\mathrm{DI})+\beta_{UD}\mu(\mathrm{UI}), \qquad \iota_U(\mu)=v_H q_\mathrm{inf}^U+\beta_{UU}\mu(\mathrm{UI})+\beta_{DU}\mu(\mathrm{DI}).$$
The continuous-time generator $Q^{\mu,a}$ this induces is discretized *exactly* via $P_{\Delta t}^{\mu,a}=\exp(\Delta t\,Q^{\mu,a})$ (`CyberSecurity.transition_probs`): unlike two-state, the transition kernel itself depends on the population law.

A computer incurs cost $f(x)=k_D\mathbf 1_{\{x\in\{\mathrm{DI,DS}\}\}}+k_I\mathbf 1_{\{x\in\{\mathrm{DI,UI}\}\}}$, a function of state alone; running and terminal rewards are both $-\Delta t\,f(x)$, discounted by $\gamma=0.5$. So, unlike two-state (where the mean-field coupling lives entirely in the reward), here it is the *dynamics* that carry all of the coupling: the reward doesn't reference $\mu$ (or even the action) at all.

**Policy.** A population-dependent 2-hidden-layer MLP (width 32, $\tanh$), taking $(t,\mu)$ and outputting a $4\times2$ logit matrix, row-softmaxed, unlike two-state's stationary lookup-table policy (`CyberSecurity.policy_probs`; all parameters packed into one flat vector so it drops into the same `action_probs_fn` machinery unchanged).

**Training/validation asymmetry.** Training uses a short horizon $T_\mathrm{train}=3$ (limits sensitivity-estimator error accumulation) with $\mu_0\sim\mathrm{Dirichlet}(1,1,1,1)$ resampled every iteration; validation freezes the policy and evaluates over a much longer $T_\mathrm{val}=50$ from the uniform law $\mu_0^\mathrm{val}=(\tfrac14,\tfrac14,\tfrac14,\tfrac14)$, deliberately feeding the policy $t$ values it never saw in training, per the reference's own protocol.

Unlike two-state, there is no known closed-form optimal policy here, so this notebook has no "vs. optimal" diagnostics (policy error, theta bias vs. ground truth); the reference itself validates qualitatively against a separately reported mean-field $Q$-learning benchmark (Carmona et al.), which is not digitized in this repo.

**This notebook shows `main`-tier results automatically whenever they're available.** `configs/cybersecurity.py`'s `MAIN` sweeps $T_\mathrm{train}\in\{3,6,12\}$ (`equal_budget`/`particle`, 5 seeds); everything below runs at $T_\mathrm{train}=3$ (matching `mid`) except the dedicated horizon-scaling section, which needs `main`'s full $T$ sweep and notes when it isn't there yet; run `scripts/train_all.sh <workers> --env cybersecurity --alg <alg> --config main` (once per algorithm) to populate it.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch
import pandas as pd

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")

from configs.cybersecurity import MAIN, MID
from mfc.environments.cybersecurity import DI, DS, UI, US, CyberSecurity, CyberSecurityConfig
from mfc.algorithms import simplex
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import set_style
from scripts.train import run_all
from scripts.test import (
    exact_gradient,
    exact_sensitivity_flow,
    generalization_eval,
    gradient_diagnostics,
    intervention_probability,
    load_runs,
    logit_perturbation_coverage,
    objective_gap,
    oracle_gradient_estimate,
    perturbation_coverage,
    rollout,
    sensitivity_estimation_error,
    state_distribution,
    state_marginal_stability,
)

set_style()

T_TRAIN_REF = MID.horizons[0]  # 3: the reference's own training horizon, and main.horizons[0]

## Configuration and budget

Auto-detects whether `runs/cybersecurity/main/` has any saved runs and uses `main` if so (5 seeds, `equal_budget`/`particle`), otherwise `mid` (1 seed, `equal_parameters`/`exact`). Both tiers use the reference's single training horizon $T_\mathrm{train}=3$; the horizon-scaling comparison lives in the two-state notebook, so the by-horizon sections below collapse to one row/curve here.

In [ ]:
main_dir = ROOT / "runs" / "cybersecurity" / "main"
tier = "main" if list(main_dir.glob("*_seed*.pt")) else "mid"
cfg = MAIN if tier == "main" else MID

print(f"tier: {tier}")
print(f"algorithms:    {cfg.algorithms}")
print(f"lambdas:       {cfg.lambdas}  (simplex perturbation scale)")
print(f"epsilon:       {cfg.epsilon}  (logit perturbation scale, fixed -- not swept like lambda)")
print(f"T_train(s)={cfg.horizons}, T_val={cfg.T_val}")
print(f"seeds:         {cfg.seeds}")
print(f"B={cfg.B}, n_aux={cfg.n_aux}, sigma={cfg.sigma}, lr={cfg.lr}, n_train={cfg.n_train}")
print(f"mu0 ~ Dirichlet(1,1,1,1) during training; validation mu0={cfg.mu0_val}")
print(f"gamma={CyberSecurityConfig().gamma}  (discount, an environment constant)")
if tier == "mid":
    print("\n(no runs/cybersecurity/main/ data yet, showing mid-tier results; run scripts/train_all.sh "
          "<workers> --env cybersecurity --alg <alg> --config main, once per algorithm, to populate the "
          "main-tier sections below)")

## Train (or load cached results)

At `mid`, trains first if nothing is cached yet. At `main`, only loads what's already there: training the full sweep is expensive and belongs in `scripts/train_all.sh`, not an inline notebook cell. Adapts `env`'s dtype to match whatever's loaded (`scripts/train.py --dtype float32` support).

In [ ]:
env = CyberSecurity()
runs_dir = ROOT / "runs" / "cybersecurity" / tier

runs = []
for alg in cfg.algorithms:
    if tier == "mid" and not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all("cybersecurity", alg, "mid")
    runs += load_runs("cybersecurity", alg, tier)

if runs and runs[0]["theta_final"].dtype != env.dtype:
    run_dtype = runs[0]["theta_final"].dtype
    print(f"note: loaded runs are {run_dtype}, switching env and the notebook's default dtype to match (was {env.dtype})")
    torch.set_default_dtype(run_dtype)
    env = CyberSecurity(dtype=run_dtype)

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded ({tier}); total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")

## Default horizon

Everything below uses $T_\mathrm{train}=3$, the only horizon either tier runs (`mid`'s, and `main.horizons[0]`). `by_lambda` picks seed 0 as a representative $\theta$ per $\lambda$; `by_lambda_all_seeds` keeps every seed for aggregate diagnostics.

In [ ]:
mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device=env.device)
gamma = env.config.gamma

T_runs = [r for r in runs if r["T"] == T_TRAIN_REF]
simplex_T_runs = [r for r in T_runs if r["alg"] == "simplex"]
mfreinforce_T_runs = [r for r in T_runs if r["alg"] == "mfreinforce"]
reinforce_T_runs = [r for r in T_runs if r["alg"] == "reinforce"]

by_lambda_all_seeds = {}
for r in simplex_T_runs:
    by_lambda_all_seeds.setdefault(r["lam"], []).append(r)
by_lambda = {lam: next((r for r in grp if r["seed"] == 0), grp[0]) for lam, grp in by_lambda_all_seeds.items()}
theta_02 = by_lambda[0.2]["theta_final"]

print(f"T_train={T_TRAIN_REF}: {len(simplex_T_runs)} simplex runs across {len(by_lambda)} lambda values, "
      f"{len(mfreinforce_T_runs)} mfreinforce run(s), {len(reinforce_T_runs)} reinforce run(s)")

## Evolution of the validation reward

The exact validation objective $J_{T_\mathrm{val}}(\theta_m;\mu_0^\mathrm{val})$ every 10 training iterations, one line per simplex $\lambda$ plus reinforce and mfreinforce (each contributing a single line, since neither sweeps a perturbation scale). At `main`, each line is the mean $\pm$ 1 std across 5 seeds.

In [ ]:
fig, ax = viz.plot_validation_curve(T_runs)
ax.set_title(f"Validation objective by training iteration ({tier} tier, T_train={T_TRAIN_REF})")

## State distribution over time (validation horizon)

The learned policy's population flow $\mu_t^\theta$ from $\mu_0^\mathrm{val}$ over $T_\mathrm{val}=50$ steps, shown for $\lambda=0.2$ (seed 0). The policy is fed $t$ values (up to 49) well beyond the $t\in\{0,1,2\}$ it ever saw during training, per the reference's own validation protocol.

In [ ]:
mu_flow = state_distribution(env, env.policy_probs, theta_02, mu0_val, cfg.T_val)
fig, ax = viz.plot_state_distribution(mu_flow, state_labels=["DI", "DS", "UI", "US"])
ax.set_title(f"Security-state population shares over the validation horizon")

## Aggregate infected/defended fractions and intervention probability

Beyond the per-state flow: the aggregate infected fraction $I_t=\mu_t(\mathrm{DI})+\mu_t(\mathrm{UI})$, defended fraction $D_t=\mu_t(\mathrm{DI})+\mu_t(\mathrm{DS})$, and the population-averaged intervention probability $A_t=\sum_x\mu_t(x)\pi_t(1\mid x,\mu_t)$ (reference "Evaluation criteria").

In [ ]:
infected, defended = env.aggregate_fractions(mu_flow)
intervention = intervention_probability(env, env.policy_probs, theta_02, mu_flow)

fig, ax = viz.plot_population_fractions({"infected I_t": infected, "defended D_t": defended, "intervention A_t": intervention})
ax.set_title("Infection, defense, and intervention shares over time")

## $J^\lambda$ vs $J$, and gradient bias/variance

The simplex plug-in gradient estimator (`mfc.algorithms.simplex.gradient_estimate`) is

$$\hat g_{B,n,\lambda,\eta}(\theta) = \frac1B\sum_{b=1}^B\sum_{t=0}^{T}\Big[\mathbb 1_{\{t<T\}}L_t^{(b)} + Q_t^{(b)}\Big]\,G_t^{(b)},$$

with $L_t=\nabla_\theta\log\pi_t^\theta(a_t\mid x_t,M_t)$ the direct policy score, $Q_t=-\frac{1-\lambda}{\lambda}H(q_t)^\top\hat D_t$ the population-sensitivity correction, and $G_t$ the return-to-go. Both diagnostics below are evaluated at each $\lambda$'s own learned $\hat\theta_\lambda$ (seed 0), at $T_\mathrm{train}$ (where the simplex machinery, and hence $J^\lambda$ and the plug-in gradient, is actually defined) rather than $T_\mathrm{val}$, discounted by $\gamma=0.5$ to match the training objective:
- $J^\lambda(\hat\theta_\lambda)$ (Monte Carlo, perturbed) vs. $J(\hat\theta_\lambda)$ (exact).
- Empirical bias/std of 30 independent draws of $\hat g(\hat\theta_\lambda)$ against the exact gradient $\nabla_\theta J(\hat\theta_\lambda)$ (autograd), summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over the full 1512-dimensional MLP parameter vector -- also shown for mfreinforce and reinforce's own (differently-defined) gradient estimators is not meaningful here, so this table stays simplex-only.

In [ ]:
gaps, grad_diag = {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    gaps[lam] = objective_gap(env, env.policy_probs, theta, mu0_val, T_TRAIN_REF, lam=lam, sigma=cfg.sigma, n_samples=5000, gamma=gamma)
    grad_diag[lam] = gradient_diagnostics(
        env, env.policy_probs, theta, mu0_val, T_TRAIN_REF,
        lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=30, gamma=gamma,
    )

rows = []
for lam in sorted(gaps):
    g, d = gaps[lam], grad_diag[lam]
    rows.append({
        "λ": lam,
        "J(theta_hat)": g["J"].item(), "J^lambda(theta_hat) (MC)": g["J_lambda_mean"].item(), "|gap|": abs(g["gap"].item()),
        "||bias||": d["bias"].norm().item(), "||std||": d["std"].norm().item(),
    })
pd.DataFrame(rows).set_index("λ")

## Perturbation coverage: simplex $d_{TV}(M^\lambda,\mu)\le\lambda$ and mfreinforce $\mathbb E[d_{TV}]\le\varepsilon/2$

Checked here (as for two-state) by direct sampling at a few representative $N=4$-dimensional population laws: the validation law, an extreme corner (all computers defended+infected), and a point from the learned flow itself. Simplex's bound holds *almost surely* (every draw); mfreinforce's logit perturbation only holds *in expectation* (`files/Discrete RL - Meunier, Pham, Reisinger.md`, Lemma 2.2).

In [ ]:
all_DI = torch.tensor([1.0, 0.0, 0.0, 0.0], dtype=env.dtype, device=env.device)
mu_labels = ["mu0_val (uniform)", "all-DI corner", "mu_flow[10]"]
mu_samples = torch.stack([mu0_val, all_DI, mu_flow[10]])

coverage = perturbation_coverage(mu_samples, lam=0.2, sigma=cfg.sigma, n_samples=5000)
for r in coverage:
    assert r["within_bound"], "the perturbation theorem's bound should never be violated"
print("simplex, lambda=0.2:")
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (lambda)": 0.2, "within_bound": r["within_bound"]}
                       for lbl, r in zip(mu_labels, coverage)]).set_index("mu"))
fig, ax = viz.plot_perturbation_coverage(coverage, 0.2, mu_labels=mu_labels)
ax.set_title("Simplex perturbation coverage against the TV bound")

logit_coverage = logit_perturbation_coverage(mu_samples, epsilon=cfg.epsilon, n_samples=5000)
for r in logit_coverage:
    assert r["within_bound"], "Lemma 2.2's expected-value bound should hold at this sample size"
print("\nmfreinforce, epsilon={}:".format(cfg.epsilon))
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (epsilon/2)": cfg.epsilon / 2, "within_bound (mean)": r["within_bound"]}
                       for lbl, r in zip(mu_labels, logit_coverage)]).set_index("mu"))

## Final objective by training horizon (main tier)

Final validation objective (at $T_\mathrm{val}$) per training horizon, every simplex $\lambda$ plus mfreinforce and reinforce, mean across `main`'s 5 seeds. `MAIN.horizons` is $(3,)$, so this is a single summary row rather than a scaling curve — widen `MAIN.horizons` to turn it into one. Requires `runs/cybersecurity/main/`.

In [ ]:
if tier != "main":
    print("main-tier data not available yet -- run scripts/train_all.sh to populate this section")
else:
    rows = []
    for T in cfg.horizons:
        row = {"T_train": T}
        for lam in cfg.lambdas:
            sel = [r for r in runs if r["alg"] == "simplex" and r["T"] == T and r["lam"] == lam]
            if sel:
                row[f"simplex λ={lam}"] = torch.tensor([r["validation_J"][-1].item() for r in sel]).mean().item()
        for alg in ("mfreinforce", "reinforce"):
            sel = [r for r in runs if r["alg"] == alg and r["T"] == T]
            if sel:
                row[alg] = torch.tensor([r["validation_J"][-1].item() for r in sel]).mean().item()
        rows.append(row)
    display(pd.DataFrame(rows).set_index("T_train"))

## Learning curves per training horizon (main tier)

The full validation-objective-vs-iteration curve at each $T_\mathrm{train}$ in `MAIN.horizons` (currently just $3$), all lambdas plus mfreinforce and reinforce. Requires `runs/cybersecurity/main/`.

In [ ]:
if tier != "main":
    print("main-tier data not available yet -- run scripts/train_all.sh to populate this section")
else:
    for T in cfg.horizons:
        sel = [r for r in runs if r["T"] == T]
        if not sel:
            continue
        fig, ax = viz.plot_validation_curve(sel)
        ax.set_title(f"Validation objective by training iteration, T_train={T}")

## Sample trajectory under the learned policy

One sampled state trajectory ($\lambda=0.2$, seed 0) from $\mu_0^\mathrm{val}$, over the training horizon $T_\mathrm{train}=3$ (states 0-3 correspond to DI, DS, UI, US). There is no known optimal policy to compare against here (unlike two-state).

In [ ]:
learned_traj = rollout(env, env.policy_probs, theta_02, mu0_val, T=T_TRAIN_REF, generator=torch.Generator(device=env.device).manual_seed(0))
fig, ax = viz.plot_trajectories(learned_traj)
ax.set_yticks([DI, DS, UI, US])
ax.set_yticklabels(["DI", "DS", "UI", "US"])
ax.set_title("Sample state path under the learned policy")

## Generalization without retraining

Evaluating every $\lambda$'s learned $\hat\theta_\lambda$ (seed 0) exactly (no retraining) under different initial laws, a longer horizon, and model misspecification (shifted infection rates / switching rate).

In [ ]:
all_US = torch.tensor([0.0, 0.0, 0.0, 1.0], dtype=env.dtype, device=env.device)
scenarios = [
    {"name": "baseline (mu0_val)"},
    {"name": "mu0=all-DI", "mu0": all_DI},
    {"name": "mu0=all-US", "mu0": all_US},
    {"name": "T=10", "T": 10},
    {"name": "stronger infection (2x q_inf)", "env": CyberSecurity(CyberSecurityConfig(q_inf_D=0.8, q_inf_U=0.6), dtype=env.dtype, device=env.device)},
    {"name": "slower switching (lambda_sw halved)", "env": CyberSecurity(CyberSecurityConfig(lambda_sw=0.4), dtype=env.dtype, device=env.device)},
]

rows = {sc["name"]: {"scenario": sc["name"]} for sc in scenarios}
for lam in sorted(by_lambda):
    gen_results = generalization_eval(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, cfg.T_val, scenarios, gamma=gamma)
    for res in gen_results:
        rows[res["name"]][f"λ={lam}"] = res["J"].item()
pd.DataFrame(list(rows.values())).set_index("scenario")

## Comparing the three algorithms

Final validation objective for each algorithm at $T_\mathrm{train}=3$: every simplex $\lambda$, reinforce (`mfc.algorithms.reinforce`, which omits the population-sensitivity correction $Q_t(D_t)$ entirely, see its module docstring), and mfreinforce (Meunier's own logit-perturbed estimator, fixed $\varepsilon$). Mean $\pm$ std across seeds at `main`; a single value at `mid` (which can't separate genuine differences from noise).

In [ ]:
by_alg = {}
for r in T_runs:
    by_alg.setdefault(r["alg"], []).append(r)

rows = []
for lam, group in sorted(by_lambda_all_seeds.items()):
    vals = torch.tensor([r["validation_J"][-1].item() for r in group])
    rows.append({"algorithm": f"simplex λ={lam}", "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0, "n_seeds": len(vals)})
for alg in ("mfreinforce", "reinforce"):
    vals = torch.tensor([r["validation_J"][-1].item() for r in by_alg[alg]])
    rows.append({"algorithm": alg, "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0, "n_seeds": len(vals)})
pd.DataFrame(rows).set_index("algorithm")

## Additional statistical validation (discrete-state theory)

The sections above check the estimators against each other and against training outcomes. The sections below instead check the theory's own asymptotic claims from `files/reference/discrete_state_space(2).tex` directly, at a single fixed $\theta=\hat\theta_{0.2}$ (`theta_02`) so $\lambda$ is the only thing varying. Unlike two-state, cybersecurity's mean-field coupling lives entirely in the **dynamics** (`transition_probs` depends on $\mu$; the reward does not depend on $\mu$ or even the action at all), so the state-marginal-stability lemma below is a genuinely non-trivial check here (twostate's own version of that lemma held with $L_K=0$ *exactly*, since twostate's dynamics don't depend on $\mu$ at all).

This MLP-policy environment is far more expensive per Monte Carlo replicate than two-state's lookup-table policy (a forward+backward pass through a 1512-parameter network per replicate, vs. closed-form derivatives), so sample sizes below are smaller than two-state's notebook uses; standard errors (SE) are still reported throughout so every point estimate's precision is explicit rather than assumed.

### Convergence of the perturbed objective: $|J^\lambda(\hat\theta_{0.2})-J(\hat\theta_{0.2})|=O(\lambda)$

Theorem "Convergence of the perturbed objective": $|J^\lambda(\theta)-J(\theta)|\le C_T\lambda$ for every $\theta$, with $C_T$ independent of $\lambda$. `n_samples=200,000` (cheap: this is a single batched Monte Carlo evaluation, not a Python-level loop).

In [ ]:
gaps_ref = {lam: objective_gap(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, lam=lam, sigma=cfg.sigma, n_samples=200_000, gamma=gamma) for lam in cfg.lambdas}

rows = []
for lam in sorted(gaps_ref):
    g = gaps_ref[lam]
    gap, se = g["gap"].item(), g["J_lambda_se"].item()
    rows.append({"λ": lam, "J(theta_02)": g["J"].item(), "J^lambda(theta_02) (MC)": g["J_lambda_mean"].item(), "SE": se, "|gap|": abs(gap), "|gap|/SE": abs(gap) / se, "|gap|/lambda": abs(gap) / lam})
pd.DataFrame(rows).set_index("λ")

### Gradient-level convergence: $\|\nabla J^\lambda(\hat\theta_{0.2})-\nabla J(\hat\theta_{0.2})\|=O(\lambda)$

Theorem "Gradient-level convergence" (needs the additional smoothness of Assumption "Smoothness of the averaged dynamics": $R_t^\theta$, $K_t^\theta$ continuously differentiable in $\mu$). The *oracle-D* plug-in estimator (`simplex.gradient_estimate` fed the exact sensitivity flow `exact_sensitivity_flow` instead of the auxiliary plug-in estimate) satisfies $\mathbb E[\hat g^{\mathrm{orc}}_{B,\lambda}(\theta)]=\nabla_\theta J^\lambda(\theta)$ exactly, isolating the perturbation-bias term from sensitivity-estimation noise. `B=cfg.B`, `reps=40` (smaller than two-state's 500, given the per-replicate MLP cost).

In [ ]:
reps_grad = 40
exact_grad_ref = exact_gradient(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, gamma=gamma)
oracle_samples_ref, oracle_mean_ref = {}, {}
for lam in cfg.lambdas:
    samples = torch.stack([
        oracle_gradient_estimate(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, lam=lam, sigma=cfg.sigma, B=cfg.B, gamma=gamma)
        for _ in range(reps_grad)
    ])
    oracle_samples_ref[lam] = samples
    oracle_mean_ref[lam] = samples.mean(dim=0)

rows = []
for lam in sorted(oracle_mean_ref):
    bias_vec = oracle_mean_ref[lam] - exact_grad_ref
    bias = bias_vec.norm().item()
    bias_se = (oracle_samples_ref[lam].std(dim=0) / reps_grad**0.5).norm().item()
    rows.append({"λ": lam, "||grad J - grad J^lambda||": bias, "SE": bias_se, "bias/SE": bias / bias_se, "bias/lambda": bias / lam})
pd.DataFrame(rows).set_index("λ")

### Population-flow sensitivity estimator: $\hat D_t(k)\to D_t^\theta(k)$

`simplex.estimate_sensitivity_flow`'s single-batch forward estimator $\hat D_t(k)$ of $D_t^\theta(k)=\nabla_\theta\mu_t^\theta(k)$, against the exact value (`exact_sensitivity_flow`, autograd). Bias $A_\eta$ is a property of $\eta$ alone (the single-batch estimator is unbiased for $D_t^{\eta,\theta}$ at *any* $n$, by the conditional-centering Remark); variance $V_\eta/n$ is a property of $n$ alone. Unlike two-state, cybersecurity's dynamics genuinely depend on $\mu$, so there is no structural reason to expect $A_\eta=0$ here.

In [ ]:
reps_sens = 150
sens_by_eta = {eta: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, eta=eta, n=50, sigma=cfg.sigma, reps=reps_sens) for eta in cfg.lambdas}
rows = [{"eta": eta, "n": 50, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "resolved (bias>2*SE)": bool(r["bias_norm"].sum().item() > 2 * r["bias_se"].sum().item())} for eta, r in sorted(sens_by_eta.items())]
print("bias vs. eta, at n=50 (large relative to cfg.n_aux=1, so V_eta/n is small and A_eta is precisely resolved):")
display(pd.DataFrame(rows).set_index("eta"))

n_values = sorted({cfg.n_aux, 5, 20, 50})
sens_by_n = {n: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, eta=0.2, n=n, sigma=cfg.sigma, reps=reps_sens) for n in n_values}
rows = [{"n": n, "eta": 0.2, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "variance": r["variance"].sum().item(), "mse": r["mse"].sum().item()} for n, r in sorted(sens_by_n.items())]
print(f"\nvariance vs. n, at fixed eta=0.2 (cfg.n_aux={cfg.n_aux} is the actual training value, an extreme low-n point):")
display(pd.DataFrame(rows).set_index("n"))

### Gradient-estimator bias decomposition: (I) Monte Carlo + (II) sensitivity-estimation + (III) perturbation

Proposition "Mean of the main-batch estimator" decomposes $\hat g_{B,n,\lambda,\eta}(\theta)-\nabla_\theta J(\theta)$ into (I) zero-mean main-batch Monte Carlo noise, (II) the bias from plugging in $\hat D_t$ instead of the exact $D_t^\theta$, and (III) the perturbation bias $\nabla J^\lambda-\nabla J$ above. The ordinary plug-in samples (`gradient_diagnostics`, `n_aux=cfg.n_aux=1`, `B=cfg.B`, `reps=40`) mix (II) and (III); the oracle-D estimator above isolates (III); their difference approximates (II).

In [ ]:
plugin_ref = {lam: gradient_diagnostics(env, env.policy_probs, theta_02, mu0_val, T_TRAIN_REF, lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=reps_grad, gamma=gamma) for lam in cfg.lambdas}

rows = []
for lam in sorted(oracle_mean_ref):
    term3 = (oracle_mean_ref[lam] - exact_grad_ref).norm().item()
    term23 = plugin_ref[lam]["bias"].norm().item()
    term2_vec = plugin_ref[lam]["mean_estimate"] - oracle_mean_ref[lam]
    term2 = term2_vec.norm().item()
    term2_se = (((plugin_ref[lam]["std"] ** 2 + oracle_samples_ref[lam].std(dim=0) ** 2) / reps_grad).sum() ** 0.5).item()
    rows.append({"λ": lam, "perturbation bias (III)": term3, "(II) approx": term2, "(II) SE": term2_se, "(II)/SE": term2 / term2_se, "total plug-in bias (II)+(III)": term23})
pd.DataFrame(rows).set_index("λ")

### Stability of the perturbed state marginal: $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)$

Lemma "Stability of the state marginal": $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)\le L_K\lambda t$, where $\nu_t^{\lambda,\theta}:=\mathrm{Law}(X_t^{\lambda,\theta})$ is the law of the *perturbed* state process (fresh $q_t$ redrawn at every step) and $\mu_t^\theta$ the exact nominal flow. Unlike two-state ($L_K=0$ exactly there, since neither dynamics nor policy depended on $\mu$), cybersecurity's transition kernel genuinely depends on $\mu$ through the infection rates, so $L_K>0$ here and this is a real, non-degenerate check: `bias`/growth in $\lambda t$ is expected. Evaluated over $T_\mathrm{val}=50$ at $\lambda$'s own learned $\hat\theta_\lambda$, `n_samples=100,000`.

In [ ]:
rows = []
for lam in sorted(by_lambda):
    tv = state_marginal_stability(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, T=cfg.T_val, lam=lam, sigma=cfg.sigma, n_samples=100_000)
    ts_to_show = [0, 1, 3, 10, 25, 50]
    rows.append({"λ": lam, **{f"t={t}": tv[t].item() for t in ts_to_show}})
pd.DataFrame(rows).set_index("λ")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")